# POC 2 project

- Creation : *28/12/2024*
- MàJ : *18/02/2025*

Réalisation d'un tentative de reproduction de la Figure 2 de l'article :
1. [ ] Modification d'un modèle AlexNet pour déconvolution
   1. [ ] Pour chaque couche similaire à son équivalente de la Figure 3
      1. [ ] Récupérer les activations choisies au hasard
1. [ ] Instanciation du modèle
1. [ ] Chargement du jeu de données
1. [ ] Calcul des déconvolutions des top 9, les champs réceptifs équivalents et les extraits des entrées correspondants
2. [ ] Affichage sous forme de grille

## Modules

In [1]:
import numpy as np
import torch
from torchvision import transforms as T
from tqdm import tqdm

In [2]:
from utils.alexnet_for_deconv import alexnetfordeconv
from utils.utils_cnn import get_output_sizes
from utils.utils_images import display_image_tensor as display_image_tensor_
from utils.topk import TopK
from datasets import DATASET_0, DATASET_2, CustomImageDataset, get_title

In [3]:
# Spécifiquement pour un carnet de type Jupyter
def display_image_tensor(img_tensor, verbose=True):
    if display:
        display_image_tensor_(img_tensor, verbose=verbose, fn_display=display)

## Devices

In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


## Les données

In [ ]:
imagenet_mean = DATASET_0["means"]
imagenet_std = DATASET_0["stds"]

geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])

transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.Lambda(lambda t: t/255.), # because read_image -> [0..255]
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])

dataset = CustomImageDataset(DATASET_2["path"], transform=transforms, extension="JPEG")

### Vérifications

In [ ]:
iter_dataset = iter(dataset)
image_t, image_trfm_t, filename = next(iter_dataset)
print(get_title(filename, DATASET_2["path"]))
display_image_tensor(image_t)
display_image_tensor(image_trfm_t)

Création d'un batch d'un image

In [ ]:
#batch_input = image_trfm_t.unsqueeze(dim=0)
batch_input = image_trfm_t.unsqueeze(dim=0).to(device)

## Le modèle

In [ ]:
model_alexnet_deconv = alexnetfordeconv(weights='IMAGENET1K_V1')
model_alexnet_deconv.eval()
model_alexnet_deconv.to(device)

Test du modèle

In [ ]:
model_alexnet_deconv(batch_input)

Taille du top K

In [ ]:
K = 9

In [ ]:
# Module indexes of layer in the paper
paper_layer_idx = [2, 5, 7, 9, 12]

rng = np.random.default_rng()
#random_layer_count = rng.integers(len(model_alexnet_deconv.features)+1)
#random_layer_idx = rng.choice(len(model_alexnet_deconv.features), random_layer_count, replace=False)

input_size=torch.Size([1, 3, 224, 224])
output_sizes = get_output_sizes(model_alexnet_deconv.features, input_size=input_size, last_2d=False)
print(output_sizes)

coord_activations = {}
for layer_idx in paper_layer_idx:
    coord_activations[layer_idx] = []
    i = 0
    while i < K:
        chn = rng.integers(output_sizes[layer_idx][0])
        row = rng.integers(output_sizes[layer_idx][1])
        col = rng.integers(output_sizes[layer_idx][2])
        if (chn, row, col) not in coord_activations[layer_idx]:
            coord_activations[layer_idx].append((chn, row, col))
            i += 1

top_activations = {i:{coord:TopK(K) for coord in coord_activations[i]} for i in paper_layer_idx}

In [ ]:
model_alexnet_deconv.features_to_device(device)
activations = model_alexnet_deconv.get_activations(batch_input, coord_activations, verbose=False)

for layer_idx, coords in activations.items():
    for coord, value in coords.items():
        top_activations[layer_idx][coord].append(value.item(), filename)

In [ ]:
for layer_idx, coords in top_activations.items():
    for coord, topk in coords.items():
        print(topk)

In [ ]:
for i in tqdm(range(len(dataset))):
    image_t, image_trfm_t, filename = dataset[i]
    batch_input = image_trfm_t.unsqueeze(dim=0).to(device)
    activations = model_alexnet_deconv.get_activations(batch_input, coord_activations, verbose=False)
    for layer_idx, coords in activations.items():
        for coord, value in coords.items():
            top_activations[layer_idx][coord].append(value.item(), filename)
    

In [ ]:
for layer_idx, coords in top_activations.items():
    for coord, topk in coords.items():
        print(layer_idx, coord, topk)

In [ ]:
from utils import all_deconvnet

In [ ]:
model_alexnet_deconv.to("cpu")

idx_layer = 2
for coord, topk in coords.items():
    #print(layer_idx, coord, topk)
    max_value, filename = topk[0]
    _, input = dataset.get_image(filename)
    batch_input = input.unsqueeze(dim=0).to(device)

    all_deconvnet(
        model_alexnet_deconv,
        batch_input,
        idx_layer=idx_layer,
        flip_kernels=False,
        use_bias=False,
        clean_feature_map=True,
        idx_map,
        verbose: bool = False
    )